# 03 – Train IsolationForest & LOF

Trains an Isolation Forest and a Local Outlier Factor (novelty=True) on the engineered feature matrix and saves them to `models/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings("ignore")


In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
CONTAMINATION = 0.08

FEAT_FILE = ROOT / "data" / "processed" / "features.parquet"
feat_df = pd.read_parquet(str(FEAT_FILE))
FEAT_COLS = [
    "requests_per_hour", "error_rate", "unique_endpoints",
    "avg_bytes_sent", "post_ratio", "is_off_hours", "is_weekend", "has_scanner_ua"
]
X = feat_df[FEAT_COLS].to_numpy(dtype=float)
print(f"Training on {X.shape[0]:,} samples × {X.shape[1]} features")


In [ ]:
# ── Fit shared scaler ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(scaler, str(MODELS_DIR / "scaler.joblib"))
print("Scaler saved.")


In [ ]:
# ── IsolationForest ───────────────────────────────────────────────────────────
iforest = IsolationForest(
    contamination=CONTAMINATION,
    n_estimators=100,
    random_state=RANDOM_STATE,
)
iforest.fit(X_scaled)

pipe_if = Pipeline([("scaler", scaler), ("iforest", iforest)])
joblib.dump(pipe_if, str(MODELS_DIR / "iforest_pipeline.joblib"))
print("IsolationForest pipeline saved.")

if_labels = iforest.predict(X_scaled)
print(f"  Anomalies flagged: {(if_labels == -1).sum()} / {len(if_labels)}")


In [ ]:
# ── LOF (novelty=True) ────────────────────────────────────────────────────────
n_neighbors = min(20, len(X_scaled) - 1)
lof = LocalOutlierFactor(
    n_neighbors=n_neighbors,
    novelty=True,
    contamination=CONTAMINATION,
)
lof.fit(X_scaled)

pipe_lof = Pipeline([("scaler", scaler), ("lof", lof)])
joblib.dump(pipe_lof, str(MODELS_DIR / "lof_pipeline.joblib"))
print("LOF pipeline saved.")

lof_labels = lof.predict(X_scaled)
print(f"  Anomalies flagged: {(lof_labels == -1).sum()} / {len(lof_labels)}")


In [ ]:
# ── Score distribution plot ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

if_scores = -iforest.decision_function(X_scaled)   # higher → more anomalous
lof_scores = -lof.decision_function(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, scores, name in zip(axes, [if_scores, lof_scores], ["IsolationForest", "LOF"]):
    ax.hist(scores, bins=40, color="steelblue", alpha=0.8)
    ax.set_title(f"{name} — Anomaly Score Distribution")
    ax.set_xlabel("Raw anomaly score (higher = more anomalous)")
    ax.set_ylabel("Count")
plt.tight_layout(); plt.show()
print("Training complete. Models saved to models/")
